In [5]:
import sys
import json
import pickle
import numpy as np
import scipy
from scipy.sparse import csr_matrix
import os
print(f"当前 Python 路径: {sys.executable}")
print(f"Scipy 版本: {scipy.__version__}")

当前 Python 路径: /root/miniconda3/envs/promptmm/bin/python
Scipy 版本: 1.15.3


In [6]:
# --- 1. 路径配置 ---
# 确保这个路径和你之前 check_data.py 里的一致
root_path = '/root/autodl-tmp/PromptMM/data/baby/' 
json_source = os.path.join(root_path, '5-core') # JSON 都在 5-core 文件夹里

In [7]:
# --- 2. 动态扫描获取真实的 N_USER 和 N_ITEM ---
max_u = 0
max_i = 0
print("正在扫描 JSON 文件以获取真实的矩阵维度...")
for json_name in ['train.json', 'val.json', 'test.json']:
    json_path = os.path.join(json_source, json_name)
    if os.path.exists(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)
            for u, items in data.items():
                max_u = max(max_u, int(u))
                if len(items) > 0:
                    max_i = max(max_i, max(items))

# ID 从 0 开始，所以真实数量必须 +1
N_USER = max_u + 1
N_ITEM = max_i + 1
print(f"🔥 扫描完毕！Baby 数据集真实维度: N_USER={N_USER}, N_ITEM={N_ITEM}")

# --- 3. 生成矩阵 ---
def generate_mat(json_name, save_name):
    json_path = os.path.join(json_source, json_name)
    save_path = os.path.join(root_path, save_name)
    
    print(f"正在读取: {json_path} ...")
    if not os.path.exists(json_path):
        print(f"❌ 错误: 找不到文件 {json_path}")
        return

    with open(json_path, 'r') as f:
        data = json.load(f)
    
    rows = []
    cols = []
    
    for u, items in data.items():
        for i in items:
            rows.append(int(u))
            cols.append(int(i))
            
    # 构建稀疏矩阵
    mat = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(N_USER, N_ITEM))
    
    with open(save_path, 'wb') as f:
        pickle.dump(mat, f)
    print(f"✅ 已生成: {save_path} (Shape: {mat.shape})")

# 执行转换
generate_mat('train.json', 'train_mat')
generate_mat('test.json', 'test_mat')

正在扫描 JSON 文件以获取真实的矩阵维度...
🔥 扫描完毕！Baby 数据集真实维度: N_USER=19445, N_ITEM=7050
正在读取: /root/autodl-tmp/PromptMM/data/baby/5-core/train.json ...
✅ 已生成: /root/autodl-tmp/PromptMM/data/baby/train_mat (Shape: (19445, 7050))
正在读取: /root/autodl-tmp/PromptMM/data/baby/5-core/test.json ...
✅ 已生成: /root/autodl-tmp/PromptMM/data/baby/test_mat (Shape: (19445, 7050))


In [8]:
# 定义根目录路径 (确保这是你的 main.py 所在路径)
project_root = '/root/autodl-tmp/PromptMM/'

# 创建 weights/sports
os.makedirs(os.path.join(project_root, 'weights/baby'), exist_ok=True)

# 创建 history/sports
os.makedirs(os.path.join(project_root, 'history/baby'), exist_ok=True)

print("✅ 文件夹已在根目录下正确创建！")

✅ 文件夹已在根目录下正确创建！
